# Introduction

In the previous notebook I analysed the correation between different models and realised for tree based models the pearson correlation was high, in this notebook I will build an ensemble model consisting Ridge and LightGBM

# Setup and loading

In [41]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

In [42]:
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge

# Load OOF predictions
MODELS_DIR = ROOT / "models"

# exclude Random Forest for now 
oof = joblib.load(MODELS_DIR / 'oof_predictions.pkl')
ridge_oof = oof['ridge_oof']
lgb_oof = oof['lgb_oof']

y = oof['y']

In [43]:
def rmse(y_true, y_pred):
  return np.sqrt(mean_squared_error(y_true, y_pred))

print('Ridge OOF RMSE:', rmse(y, ridge_oof))
print('LGB OOF RMSE:', rmse(y, lgb_oof))


Ridge OOF RMSE: 0.14701048328624955
LGB OOF RMSE: 0.13685615941445264


# Testing Ensembles

In [44]:
oof_matrix = np.column_stack([
  ridge_oof,
  lgb_oof
])

In [45]:
# Optimise ensemble weights 
from scipy.optimize import minimize

def ensemble_rmse(weights):
  blended = np.dot(oof_matrix, weights)
  return rmse(y, blended)

constraints = {'type':'eq', 'fun':lambda w : np.sum(w)-1}
bounds = [(0,1), (0,1)]
initial_weights = np.array([0.5,0.5])

result = minimize(
  ensemble_rmse,
  initial_weights,
  bounds=bounds,
  constraints=constraints
)

best_weights = result.x
best_rmse= ensemble_rmse(best_weights)

print('Best weights [Ridge, LGB]:', best_weights)
print('Ensemble OOF RMSE:', best_rmse)

Best weights [Ridge, LGB]: [0.3440155 0.6559845]
Ensemble OOF RMSE: 0.13279724990095323


The ensemble consisting Ridge and LGB indeed yields good RMSE, but now I will try what if I make an ensemble of all the three models (Ridge, LGBM, and RandomForest)

In [46]:
rfr_oof = oof['rfr_oof']

oof_matrix_all = np.column_stack([
  ridge_oof,
  lgb_oof,
  rfr_oof
])


def ensemble_rmse_all(weights):
  blended = np.dot(oof_matrix_all, weights)
  return rmse(y, blended)

constraints = {'type':'eq', 'fun':lambda w : np.sum(w)-1}
bounds = [(0,1), (0,1), (0,1)]
initial_weights = np.array([1/3,1/3,1/3])

result_all = minimize(
  ensemble_rmse_all,
  initial_weights,
  bounds=bounds,
  constraints=constraints
)

best_weights_all = result_all.x
best_rmse_all= ensemble_rmse_all(best_weights_all)

print('Best weights [Ridge, LGB,RFR]:', best_weights_all)
print('Ensemble OOF RMSE (all three models):', best_rmse_all)

Best weights [Ridge, LGB,RFR]: [0.34196129 0.49700167 0.16103704]
Ensemble OOF RMSE (all three models): 0.13252223810933014


The three model ensemble did yield some improvement but it is very small, about 0.00028, this proves what was concluded from model correlation analysis in the error analysis notebook. LightGBM and Random Forest are all tree based models and they are highly correlated, most ensemble benefits are already captured by adding LightGBM, adding Random Forest would only have tiny gains.

Since adding RandomForest would only add tiny improvement, Random Forest will be excluded from the ensemble for clarity 

# Final Prediction

In [47]:
FINAL_MODELS = ['ridge', 'lgb']

final_weights = best_weights
print('Final ensemble models:', FINAL_MODELS)
print('Final ensemble weights:', final_weights)

Final ensemble models: ['ridge', 'lgb']
Final ensemble weights: [0.3440155 0.6559845]


In [48]:
# Retrain the final ensemble model on the full dataset 
from src.data import load_train_data
from src.features import add_engineered_features

# Load full training data
train_df = load_train_data()
X_full = add_engineered_features(train_df)
y_full = np.log1p(train_df['SalePrice'])

# Refit the two selected models
ridge_full = joblib.load(MODELS_DIR /  'ridge_final.pkl')
ridge_full.fit(X_full,y_full)

lgb_full = joblib.load(MODELS_DIR /  'lgbm_final.pkl')
lgb_full.fit(X_full,y_full)

# Save final trained models 
joblib.dump(ridge_full, MODELS_DIR/'ridge_full.pkl')
joblib.dump(lgb_full, MODELS_DIR/'lgb_full.pkl')
print('models saved')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001483 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2927
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 199
[LightGBM] [Info] Start training from score 12.024057
models saved


In [50]:
from src.data import load_test_data

test_df = load_test_data()
X_test = add_engineered_features(test_df)
test_ids = test_df['Id'].values

ridge_test_pred = ridge_full.predict(X_test)
lgb_test_pred = lgb_full.predict(X_test)

# apply ensemble weights
final_test_pred_log = (
  final_weights[0] * ridge_test_pred +
  final_weights[1] * lgb_test_pred
)

# convert back from log
final_test_pred = np.expm1(final_test_pred_log)

In [51]:
# Submission file 
submission = pd.DataFrame({
  'Id': test_ids,
  'SalePrice': final_test_pred
})

submission_path = ROOT / 'submissions'/ 'submission.csv'
submission.to_csv(submission_path, index= False)

submission.head()

,Id,SalePrice
0,1461,120247.855262
1,1462,156280.820992
2,1463,177325.915483
3,1464,191377.836142
4,1465,189858.315315
